### Import

In [1]:
import torch
import torchvision
print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open("prompt.txt", "r", encoding="utf-8") as f:
    instruction= f.read()

# Cuz Titron is a chud
import os
cl_dir = r"C:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.51.36231\bin\Hostx64\x64"
os.environ["PATH"] = cl_dir + ";" + os.environ["PATH"]
os.environ["CC"] = cl_dir + r"\cl.exe"
os.environ["CXX"] = cl_dir + r"\cl.exe"

Torch: 2.8.0+cu126
TorchVision: 0.23.0+cu126
Using device: cuda


In [2]:
import unsloth
print(unsloth.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.
C:\Users\yujie.lim\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0908 16:06:22.237000 24764 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
2026.8.22


In [ ]:
import json
from pathlib import Path
from unsloth import FastVisionModel

# model, processor = FastVisionModel.from_pretrained(
#     model_name="unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
#     load_in_4bit=True,
#     use_gradient_checkpointing="unsloth",
# )

# model = FastVisionModel.for_training(model)
# print(f"Model ready on {device}")

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 729/729 [00:03<00:00, 190.95it/s]


Model ready on cuda


### Train-test-split

In [ ]:
from datasets import Dataset
from PIL import Image

# Change according to number on the last KAD
no_of_kads= 2

# for range purpose, don't change
no_of_kads+= 1

data = {
    "image": [
          Image.open(f"images/myKad{i}.jpg") for i in range(no_of_kads)
],
    "text": [
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['image', 'text'],
    num_rows: 3
})


In [6]:

# Split it - 80/20 is standard, adjust if needed
train_test = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test['train']
test_dataset = train_test['test']

# For QLoRA fine-tuning with Qwen-VL, you'll need a collate function
def collate_fn(batch):
    images = [item['image'] for item in batch]
    texts = [item['text'] for item in batch]
    # Process based on your model's processor
    return {'images': images, 'texts': texts}

# Then pass to DataLoader
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, collate_fn=collate_fn)

In [7]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.visual`: `get_input_embeddings` not auto‑handled for Qwen3_5VisionModel; please override in the subclass.. Falling back to pre-forward hook.


In [8]:
def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : sample["image"]} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["text"]} ]
        },
    ]
    return { "messages" : conversation }
pass

In [9]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image = dataset[2]["image"]

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

[unsloth_zoo.log|WARNING]Unsloth: torch.compile could not generate code for Qwen3_5VisionPatchEmbed_forward; running it eagerly from here. Training is unaffected apart from speed. (BackendCompilerFailed)
[unsloth_zoo.log|WARNING]Unsloth: torch.compile could not generate code for Qwen3_5VisionRotaryEmbedding_forward; running it eagerly from here. Training is unaffected apart from speed. (BackendCompilerFailed)
[unsloth_zoo.log|WARNING]Unsloth: torch.compile could not generate code for apply_rotary_pos_emb_vision; running it eagerly from here. Training is unaffected apart from speed. (BackendCompilerFailed)
[unsloth_zoo.log|WARNING]Unsloth: torch.compile could not generate code for rotate_half; running it eagerly from here. Training is unaffected apart from speed. (BackendCompilerFailed)
[unsloth_zoo.log|WARNING]Unsloth: torch.compile could not generate code for Qwen3_5RMSNorm_forward; running it eagerly from here. Training is unaffected apart from speed. (BackendCompilerFailed)
[unsloth

```json
{
    "Id-no": "040825-13-075",
    "Name": "LUZMAN AL THAQIF BIN",
    "Address1": "NARZARUDDIN",
    "Address2": "FLAT SUNGAI RAJANG",
    "Address3": "JALAN REPOK",
    "Postal_Code": "96100 SARIKEI",
    "State": "SARAWAK",
    "Gender": "K"
}
```<|im_end|>
<|endoftext|>


### Training 

In [11]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


In [12]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 4060 Laptop GPU. Max memory = 7.996 GB.
2.295 GB of memory reserved.


In [13]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 30 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,181,952 of 866,167,872 (1.52% trained)


Step,Training Loss
1,1.681357
2,1.681357
3,1.620887
4,1.483337
5,1.262102
6,1.015645
7,0.779433
8,0.682551
9,0.459721
10,0.342577


Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-30\tokenizer_config.json.


In [14]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

256.1571 seconds used for training.
4.27 minutes used for training.
Peak reserved memory = 5.561 GB.
Peak reserved memory for training = 3.266 GB.
Peak reserved memory % of max memory = 69.547 %.
Peak reserved memory for training % of max memory = 40.845 %.


In [19]:
FastVisionModel.for_inference(model) # Enable for inference!

image = Image.open("images/myKad34.jpg")

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

{
          "Id-no": "810824-13-5122",
          "Name": "LIEW NYUK PHIN",
          "Address1": "1497 LORONG 8",
          "Address2": "JALAN JEE FOH",
          "Address3": "KROKOP",
          "Postal_Code": "98000 MIRI",
          "State": "SARAWAK",
          "Gender": "PEREMPUAN"
        }<|im_end|>
